In [ ]:
# @title Upload ApacheOne/SD15_MACaQ_pruned_fullwidth
# Paste this cell below Cell 3 in your completed Colab session.
# After a session reset, restore the complete run folder and point to the original FP16 checkpoint.
RUN_FOLDER = "/content/sd15_macaq_fullwidth_from_fp16/f600083329dd/9a5174a1923e"
ORIGINAL_FP16 = ""  # Optional fallback if the completed run's pipeline is unavailable
PRIVATE_REPO = True  # Set False to create a public model repo
UPLOAD = True  # Set False to generate and review the local upload folder
HF_TOKEN_SECRET = "HF_TOKEN"
WEIGHTS_LICENSE_FILE = ""  # Optional path to the applicable checkpoint license text

import hashlib
import html
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from getpass import getpass
from pathlib import Path

REPO_ID = "ApacheOne/SD15_MACaQ_pruned_fullwidth"
SOURCE_SHA = "1a16a9524427019dc513b398087035e7e36ad2f2aa9691bd71d28ad959e94180"
SOURCE_FILE = "realismByStableYogi_sd15V9LCM.safetensors"
SOURCE_PAGE = "https://civitai.com/models/166609/realism-by-stable-yogi-pony?modelVersionId=1235013"
AUTHOR_PAGE = "https://huggingface.co/Stableyogi/Realism-Pony-Checkpoints"
BASE_CONFIG = "stable-diffusion-v1-5/stable-diffusion-v1-5"
BT = chr(96)

def read_json(path):
    return json.loads(path.read_text())

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")

def copy_file(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

def safetensors_header(path):
    with path.open("rb") as f:
        length = int.from_bytes(f.read(8), "little")
        entries = json.loads(f.read(length))
    tensors = {}
    for name, entry in entries.items():
        if name != "__metadata__":
            first, last = entry["data_offsets"]
            tensors[name] = dict(shape=entry["shape"], storage_dtype=entry["dtype"], bytes=last - first)
    if length + 8 + sum(t["bytes"] for t in tensors.values()) != path.stat().st_size:
        raise ValueError(f"Safetensors payload mismatch in {path}")
    return tensors

def copy_runtime(source, destination):
    if not source.is_dir():
        raise FileNotFoundError(source)
    shutil.copytree(source, destination, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git"))

def layer_group(name, shape):
    if shape and len(shape) == 4:
        return "3×3 convolutions" if shape[-2:] == [3, 3] else "1×1 convolutions" if shape[-2:] == [1, 1] else "Other convolutions"
    if ".attn1." in name or ".attn2." in name:
        return "Attention projections"
    if ".ff." in name:
        return "MLP projections"
    return "Normalization" if shape and len(shape) == 1 else "Timestep and other linears"

def make_inventory(unet_path, specs):
    grouped = defaultdict(list)
    for name, tensor in sorted(safetensors_header(unet_path).items()):
        grouped[name.rpartition(".")[0]].append(dict(name=name, **tensor))
    layers = {}
    for name, tensors in grouped.items():
        spec = specs.get(name)
        shape = spec["weight_shape"] if spec else next(
            (t["shape"] for t in tensors if t["name"] == name + ".weight"), None
        )
        layers[name] = dict(
            precision="MXFP4 W4A4" if spec else "/".join(sorted({t["storage_dtype"] for t in tensors})),
            original_weight_shape=shape,
            stored_bytes=sum(t["bytes"] for t in tensors),
            tensors=tensors,
        )
    for name in specs:
        if not any(t["name"] == name + ".W_packed" for t in layers.get(name, {}).get("tensors", [])):
            raise ValueError(f"Packed weights missing for {name}")
    return layers

PIPELINE_LOADER_SOURCE = '"""Load the uploaded SD 1.5 MACaQ model and its bundled components."""\nimport sys\nfrom pathlib import Path\n\nimport torch\nfrom diffusers import AutoencoderKL, LCMScheduler, StableDiffusionPipeline\nfrom huggingface_hub import snapshot_download\nfrom transformers import CLIPTextModel, CLIPTokenizer\n\n\ndef load_pipeline(repo_or_directory, device="cuda", backend="native", token=None):\n    directory = Path(repo_or_directory)\n    if not directory.is_dir():\n        directory = Path(snapshot_download(repo_or_directory, token=token))\n    sys.path.insert(0, str(directory / "unet" / "runtime"))\n    from macaq_unet_conv import load_macaq_unet\n\n    unet = load_macaq_unet(directory / "unet", backend=backend, device=device)\n    encoder_dir = directory / "text_encoder"\n    if (encoder_dir / "manifest.json").is_file():\n        sys.path.insert(0, str(encoder_dir / "runtime"))\n        from load_macaq_clip import load_macaq_clip\n\n        text_encoder = load_macaq_clip(encoder_dir, backend=backend, device=device)\n    else:\n        text_encoder = CLIPTextModel.from_pretrained(encoder_dir, torch_dtype=torch.float16).eval().to(device)\n    vae = AutoencoderKL.from_pretrained(directory / "vae", torch_dtype=torch.float16).eval().to(device)\n    tokenizer = CLIPTokenizer.from_pretrained(directory / "tokenizer", local_files_only=True)\n    scheduler = LCMScheduler.from_pretrained(directory / "scheduler", local_files_only=True)\n    return StableDiffusionPipeline(\n        vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, unet=unet,\n        scheduler=scheduler, safety_checker=None, feature_extractor=None,\n        requires_safety_checker=False,\n    )\n'


def release_inputs(run):
    unet = run / "unet"
    report = run / "report"
    required = [
        unet / "model.safetensors", unet / "manifest.json", unet / "config.json",
        unet / "runtime" / "macaq_unet_conv.py", unet / "runtime" / "sd15_macaq_runtime.py",
        report / "fullwidth_from_fp16.json",
    ]
    for path in required:
        if not path.is_file():
            raise FileNotFoundError(f"Missing {path}. Use the runtime that completed Cell 3 or restore the run folder.")
    manifest = read_json(unet / "manifest.json")
    if (
        manifest.get("format") != "sd15-macaq-full-unet-conv-v1"
        or manifest.get("source_sha256") != SOURCE_SHA
        or manifest.get("conv_hessian_block") != "full_per_layer"
        or manifest.get("linear_hessian_block") != "full_per_layer"
    ):
        raise ValueError("Expected the full-width Stable Yogi SD 1.5 V9 LCM export from Cell 3.")
    metrics = read_json(report / "fullwidth_from_fp16.json")
    fits = manifest["layer_fits"]
    specs = manifest["layers"]
    if set(fits) != set(specs) or len(fits) != metrics["packed_layers"] or len(fits) != 282:
        raise ValueError("The run must contain all 282 layer fits.")
    if (
        fits != metrics["layer_fits"]
        or manifest["unet_tensor_bytes"] != metrics["unet_tensor_bytes"]
        or manifest["original_unet_parameter_bytes"] != metrics["original_unet_bytes"]
        or not metrics["images"]
    ):
        raise ValueError("U-Net manifest and comparison report disagree.")
    for name, fit in fits.items():
        width = specs[name]["in_features"]
        if (
            fit["input_width"] != width
            or fit["hessian_width"] != math.ceil(width / 32) * 32
            or fit["rows_converged"] != specs[name]["out_features"]
        ):
            raise ValueError(f"Layer fit metadata mismatch for {name}.")
    return manifest, metrics


def release_components(run, live_pipe, live_model_dir, checkpoint_path, stage):
    import torch

    if (
        live_pipe is not None and live_model_dir is not None
        and Path(live_model_dir).resolve() == (run / "unet").resolve()
        and not ORIGINAL_FP16
    ):
        components = live_pipe
    else:
        candidates = [Path(ORIGINAL_FP16).expanduser()] if ORIGINAL_FP16 else [
            *([Path(checkpoint_path).expanduser()] if checkpoint_path else []),
            run.parent.parent / SOURCE_FILE,
            run / SOURCE_FILE,
        ]
        source = next((path for path in candidates if path.is_file()), None)
        if source is None:
            raise FileNotFoundError("Set ORIGINAL_FP16 to the original checkpoint to export CLIP and VAE.")
        with source.open("rb") as handle:
            if hashlib.file_digest(handle, "sha256").hexdigest() != SOURCE_SHA:
                raise ValueError("ORIGINAL_FP16 SHA-256 differs from the source used in this run.")
        from diffusers import StableDiffusionPipeline

        components = StableDiffusionPipeline.from_single_file(
            str(source), config=BASE_CONFIG, torch_dtype=torch.float16,
            safety_checker=None, feature_extractor=None, requires_safety_checker=False,
        )
    components.text_encoder.save_pretrained(stage / "text_encoder", safe_serialization=True)
    components.vae.save_pretrained(stage / "vae", safe_serialization=True)
    components.tokenizer.save_pretrained(stage / "tokenizer")


def model_card(manifest, metrics, inventory, license_ref):
    layers = inventory["unet_layers"]
    fits = manifest["layer_fits"]
    sources = inventory["component_tensor_bytes"]
    precision = inventory["component_storage_dtype"]
    reduction = inventory["unet_storage_reduction_percent"]
    compressed = metrics["unet_tensor_bytes"]
    baseline = metrics["original_unet_bytes"]
    linear_count = sum(row["kind"] == "linear" for row in fits.values())
    conv_count = sum(row["kind"] == "conv2d" for row in fits.values())
    widest = max(row["hessian_width"] for row in fits.values())
    lpips = metrics["mean_lpips_to_fp16"]
    runtime = (
        "The quantized column used emulated MXFP4 W4A4 on the L4, with FP4 rounding and BF16 matrix multiplication."
        if metrics["runtime"] == "emulated" else "The quantized column used native MXFP4 W4A4."
    )
    readme = f"""---
license: other
license_name: stable-yogi-source-terms
license_link: "{license_ref}"
pipeline_tag: text-to-image
tags:
- stable-diffusion
- sd-1.5
- lcm
- macaq
- mxfp4
---

# SD15 MACaQ Pruned Full-Width

**{reduction:.2f}% smaller U-Net weight payload** · {baseline / 1e6:,.2f} MB FP16 → **{compressed / 1e6:,.2f} MB ({compressed / 2**20:.2f} MiB)** packed.

This is an MXFP4 W4A4 quantization of Stable Yogi SD 1.5 V9 LCM. It retains the original U-Net layer dimensions. The repository name follows the earlier MACaQ release. The storage savings come from weight quantization, and all U-Net layers remain present.

## Weights

| Component | Precision | Tensor payload |
|---|---|---:|
| U-Net | MXFP4 W4A4 with retained higher-precision tensors | {compressed / 1e6:.2f} MB |
| CLIP text encoder | {precision["text_encoder"]} | {sources["text_encoder"] / 1e6:.2f} MB |
| VAE | {precision["vae"]} | {sources["vae"] / 1e6:.2f} MB |
| **Total weight files including headers** | | **{inventory["weight_file_bytes"] / 1e6:.2f} MB** |

## Full-width calibration

The U-Net was refit from the original FP16 checkpoint. Calibration sampled activations from **{len(manifest["calibration_prompts"])} prompts** with up to **{manifest["rows_per_step"]} rows per forward call**. Each of the **{linear_count} linear layers and {conv_count} convolutions** used a Hessian that spans its full input width, including flattened convolution patch channels, with alignment padding to multiples of 32 where needed. The widest layer used **{widest:,} inputs**. The earlier release used 256-wide convolution Hessian blocks.

The fit balances input channels, applies a blockwise Hadamard rotation, screens block exponents, then refines adjacent FP4 weight codes using the full Hessian at fixed exponents. Each row is refined to a local stopping criterion. Per-layer sample counts, widths, flip counts, and objectives are in the [run report](fullwidth_from_fp16.json).

## FP16 comparisons

Six matched prompt-and-seed pairs were rendered at **{metrics["width"]}×{metrics["height"]}** using **LCM, {metrics["steps"]} steps, and CFG {metrics["cfg"]:g}**. The left image is FP16 and the right image uses the quantized U-Net. Both use the same CLIP text encoder and VAE. {runtime}

The mean LPIPS to FP16 was **{lpips:.4f}** across these six pairs. Lower values indicate closer image appearance. Some details change, including glasses in the train-station scene and the car's viewing angle. The earlier release's results were reported with SSIM, so this LPIPS result does not rank the two releases.

"""
    for row in metrics["images"]:
        prompt = html.escape(row["prompt"]).replace("\n", " ")
        readme += (
            f'**Seed {row["seed"]}** · {prompt} · LPIPS {row["lpips_to_fp16"]:.4f}\n\n'
            f'![FP16 and MXFP4 images for seed {row["seed"]}](comparisons/{row["image"]})\n\n'
        )
    groups = defaultdict(list)
    for name, info in layers.items():
        groups[layer_group(name, info["original_weight_shape"])].append((name, info))
    lines = [
        "## U-Net layer inventory", "",
        "The [JSON inventory](sd15_macaq_unet_layers.json) includes every stored U-Net tensor.",
        "", "| Layer group | Modules | Storage (MiB) |", "|---|---:|---:|",
    ]
    for group, entries in groups.items():
        size = sum(info["stored_bytes"] for _, info in entries)
        lines.append(f"| {group} | {len(entries)} | {size / 2**20:.2f} |")
    for group, entries in groups.items():
        lines += [
            "", "<details>", f"<summary>{group} · {len(entries)} layers</summary>", "",
            "| Layer | Precision | Original weight shape | Hessian width | Stored bytes | Tensor dtypes and shapes |",
            "|---|---|---|---:|---:|---|",
        ]
        for name, info in entries:
            shape = "×".join(map(str, info["original_weight_shape"])) if info["original_weight_shape"] else "—"
            hessian = f"{fits[name]['hessian_width']:,}" if name in fits else "—"
            tensors = ", ".join(
                f"{BT}{t['name'].rsplit('.', 1)[-1]}{BT} {t['storage_dtype']} "
                f"[{'×'.join(map(str, t['shape'])) or 'scalar'}]" for t in info["tensors"]
            )
            lines.append(f"| {BT}{name}{BT} | {info['precision']} | {shape} | {hessian} | {info['stored_bytes']:,} | {tensors} |")
        lines.extend(["", "</details>"])
    readme += "\n".join(lines) + "\n\n"
    fence = BT * 3
    return readme + f"""## Run locally

Use CUDA PyTorch with Triton, Diffusers 0.39.0, Transformers 4.57.6, Accelerate, Safetensors, and Hugging Face Hub. The packed U-Net uses the included custom loader. Use {BT}backend="native"{BT} for native MXFP4 on a compatible Blackwell setup, or {BT}backend="emulated"{BT} to reproduce the L4 evaluation. Native inference was outside the reported test.

{fence}python
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

directory = Path(snapshot_download("{REPO_ID}"))
sys.path.insert(0, str(directory))
from load_pipeline import load_pipeline

torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
pipe = load_pipeline(directory, device="cuda", backend="emulated")
image = pipe(
    "A ceramic teapot on a wooden table",
    height={metrics["height"]}, width={metrics["width"]},
    num_inference_steps={metrics["steps"]}, guidance_scale={metrics["cfg"]:g},
    generator=torch.Generator("cuda").manual_seed(7003),
).images[0]
image.save("macaq_fullwidth.png")
{fence}

## Sources and terms

- [Stable Yogi SD 1.5 V9 LCM source checkpoint, version 1235013]({SOURCE_PAGE}).
- [Stable Yogi checkpoint collection and source terms]({AUTHOR_PAGE}).
- [MACaQ method and reference runtime](https://github.com/huawei-csl/MACaQ). Runtime license and notices are under {BT}unet/runtime/macaq_vendor{BT}.
- [Earlier MACaQ release](https://huggingface.co/ApacheOne/SD15_MACaQ_pruned).

Source checkpoint SHA-256 is {BT}{SOURCE_SHA}{BT}.
"""


def stage_release(run, live_pipe, live_model_dir, checkpoint_path):
    manifest, metrics = release_inputs(run)
    packed_source = run / "unet" / "model.safetensors"
    layers = make_inventory(packed_source, manifest["layers"])
    unet_bytes = sum(info["stored_bytes"] for info in layers.values())
    if unet_bytes != metrics["unet_tensor_bytes"]:
        raise ValueError("Packed U-Net payload differs from the report.")

    stage = Path(tempfile.mkdtemp(prefix="sd15_macaq_fullwidth_hf_", dir="/content" if Path("/content").is_dir() else None))
    try:
        for filename in ("model.safetensors", "config.json", "manifest.json"):
            copy_file(run / "unet" / filename, stage / "unet" / filename)
        copy_runtime(run / "unet" / "runtime", stage / "unet" / "runtime")
        release_components(run, live_pipe, live_model_dir, checkpoint_path, stage)
        write_json(stage / "scheduler" / "scheduler_config.json", manifest["scheduler_config"])
        (stage / "load_pipeline.py").write_text(PIPELINE_LOADER_SOURCE, encoding="utf-8")
        copy_file(run / "report" / "fullwidth_from_fp16.json", stage / "fullwidth_from_fp16.json")
        for row in metrics["images"]:
            filename = row["image"]
            if Path(filename).name != filename or Path(filename).suffix.lower() != ".png":
                raise ValueError(f"Unexpected comparison image name {filename!r}.")
            copy_file(run / "report" / filename, stage / "comparisons" / filename)
        license_ref = SOURCE_PAGE
        if WEIGHTS_LICENSE_FILE:
            copy_file(Path(WEIGHTS_LICENSE_FILE).expanduser(), stage / "LICENSE")
            license_ref = "LICENSE"

        weight_paths = {
            "unet": stage / "unet" / "model.safetensors",
            "text_encoder": stage / "text_encoder" / "model.safetensors",
            "vae": stage / "vae" / "diffusion_pytorch_model.safetensors",
        }
        bytes_by_component = {"unet": unet_bytes}
        dtypes = {}
        for component in ("text_encoder", "vae"):
            tensors = safetensors_header(weight_paths[component])
            bytes_by_component[component] = sum(row["bytes"] for row in tensors.values())
            storage = {row["storage_dtype"] for row in tensors.values()}
            dtypes[component] = "FP16" if storage == {"F16"} else "/".join(sorted(storage))
        inventory = dict(
            original_fp16_unet_bytes=metrics["original_unet_bytes"],
            unet_storage_reduction_percent=100 * (1 - unet_bytes / metrics["original_unet_bytes"]),
            component_tensor_bytes=bytes_by_component,
            component_storage_dtype=dtypes,
            weight_file_bytes=sum(path.stat().st_size for path in weight_paths.values()),
            unet_quantized_modules=len(manifest["layers"]),
            unet_layers=layers,
        )
        write_json(stage / "sd15_macaq_unet_layers.json", inventory)
        (stage / "README.md").write_text(model_card(manifest, metrics, inventory, license_ref), encoding="utf-8")
    except Exception:
        shutil.rmtree(stage)
        raise
    return stage


def push_release(stage):
    from huggingface_hub import HfApi

    token = os.environ.get(HF_TOKEN_SECRET, "").strip()
    if not token:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                token = (userdata.get(HF_TOKEN_SECRET) or "").strip()
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    if not token:
        token = getpass(f"Hugging Face write token ({HF_TOKEN_SECRET}): ").strip()
    if not token:
        raise ValueError("A Hugging Face write token for the ApacheOne repo is required.")

    api = HfApi(token=token)
    api.create_repo(repo_id=REPO_ID, repo_type="model", private=PRIVATE_REPO, exist_ok=True)
    commit = api.upload_folder(
        repo_id=REPO_ID, repo_type="model", folder_path=str(stage),
        commit_message="Publish the full-width SD 1.5 MACaQ model and FP16 comparisons",
    )
    expected = {path.relative_to(stage).as_posix() for path in stage.rglob("*") if path.is_file()}
    actual = set(api.list_repo_files(repo_id=REPO_ID, repo_type="model", revision=commit.oid))
    if missing := expected - actual:
        raise RuntimeError(f"Files missing in {commit.commit_url}: {sorted(missing)}")
    print(f"Verified {len(expected)} files at https://huggingface.co/{REPO_ID}")
    print(f"Commit: {commit.commit_url}")


def publish_fullwidth():
    run = Path(RUN_FOLDER).expanduser()
    live_model_dir = globals().get("model_dir")
    if not (run / "unet" / "manifest.json").is_file() and live_model_dir:
        live_dir = Path(live_model_dir)
        if (live_dir / "manifest.json").is_file():
            run = live_dir.parent
    stage = stage_release(run, globals().get("pipe"), live_model_dir, globals().get("CHECKPOINT_PATH"))
    print(f"Staged upload: {stage}")
    return stage


# Colab already has PyTorch. Install supporting packages only in a fresh session.
if "pipe" not in globals() or globals()["pipe"] is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "diffusers==0.39.0", "transformers==4.57.6", "accelerate>=1.0",
        "safetensors>=0.8", "huggingface_hub>=0.36,<2", "ftfy",
    ])
fullwidth_upload_dir = publish_fullwidth()
if UPLOAD:
    push_release(fullwidth_upload_dir)
